# Neo4j Векторный поиск
1. Connects to Neo4j using the configured URI/user/password.
2. Creates a vector index on `Encounter.embedding_e5` (индекс на уровне бд по свойству ...)
3. Builds text cards for each `Encounter` (cities/airports/countries/documents/topics).
4. Computes E5 embeddings for those cards.
5. Writes embeddings back to Neo4j as a node property.


# Создание эмбедингов для Encounter через карточки

In [24]:
# 1 создание эмбедингов для Encounters через карточки + инфа о связи bva-paris и аналоги
# Подключение к БД
from neo4j import GraphDatabase

NEO4J_URI = "neo4j://localhost:7689"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"
NEO4J_DB = "neo4j"

EMBEDDING_PROP = "embedding_e5"
VECTOR_INDEX_NAME = "encounter_embedding_e5"
VECTOR_DIM = 768  # e5-base-v2

def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Создание векторного индекса по свойству Encounter.embedding_e5 
def ensure_vector_index(driver):
    with driver.session(database=NEO4J_DB) as session:
        session.run(
            f"""
            CREATE VECTOR INDEX {VECTOR_INDEX_NAME} IF NOT EXISTS
            FOR (e:Encounter) ON (e.{EMBEDDING_PROP})
            OPTIONS {{ indexConfig: {{ `vector.dimensions`: {VECTOR_DIM}, `vector.similarity_function`: 'cosine' }} }}
            """
        ).consume()
from typing import Dict, List

# Map smaller/metro airports to their larger city for embedding context
AIRPORT_ALIAS_CITY = {
    "airport_bva": "Paris",
    "airport_cdg": "Paris",
    "airport_ory": "Paris",
    "airport_sdg": "Paris",
    "airport_memmingen": "Munich",
    "airport_malpensa": "Milan",
    "airport_marco_polo": "Venice",
    "airport_fiumicino": "Rome",
    "airport_tegel": "Berlin",
    "airport_schonefeld": "Berlin",
    "airport_pisa": "Florence",
}

def fetch_encounter_cards(driver) -> Dict[str, str]:
    query = '''
    MATCH (e:Encounter)
    OPTIONAL MATCH (e)-[:atCity]->(city:City)
    OPTIONAL MATCH (e)-[:atAirport]->(airport:Airport)
    OPTIONAL MATCH (e)-[:atCountry]->(country:Country)
    OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
    OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
    RETURN e.key AS ekey,
           collect(DISTINCT city.key) AS cities,
           collect(DISTINCT airport.key) AS airports,
           collect(DISTINCT country.key) AS countries,
           collect(DISTINCT di.documentType) AS doc_types,
           collect(DISTINCT q.topicKey) AS topics
    '''
    cards = {}
    with driver.session(database=NEO4J_DB) as session:
        for r in session.run(query):
            ekey = r["ekey"]
            if not ekey:
                continue
            parts = [f"Encounter: {ekey}"]
            if r["cities"]:
                parts.append("Cities: " + ", ".join(sorted([c for c in r["cities"] if c])))
            if r["airports"]:
                airports = sorted([a for a in r["airports"] if a])
                parts.append("Airports: " + ", ".join(airports))
                alias_cities = sorted({AIRPORT_ALIAS_CITY[a] for a in airports if a in AIRPORT_ALIAS_CITY})
                if alias_cities:
                    parts.append("AliasCities: " + ", ".join(alias_cities))
            if r["countries"]:
                parts.append("Countries: " + ", ".join(sorted([c for c in r["countries"] if c])))
            if r["doc_types"]:
                parts.append("Documents: " + ", ".join(sorted([d for d in r["doc_types"] if d])))
            if r["topics"]:
                parts.append("Topics: " + ", ".join(sorted([t for t in r["topics"] if t])))
            cards[ekey] = " | ".join(parts)
    return cards

from typing import Iterable
try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError("Install sentence-transformers to compute E5 embeddings") from e

def embed_texts_e5(model: SentenceTransformer, texts: Iterable[str], is_query: bool = False) -> List[List[float]]:
    prefix = "query: " if is_query else "passage: "
    texts = [prefix + t for t in texts]
    vecs = model.encode(texts, normalize_embeddings=True)
    return [v.tolist() for v in vecs]

def build_embeddings_for_encounters(cards: Dict[str, str]):
    model = SentenceTransformer("intfloat/e5-base-v2")
    keys = list(cards.keys())
    texts = [cards[k] for k in keys]
    vectors = embed_texts_e5(model, texts, is_query=False)
    return keys, vectors
def write_embeddings(driver, keys: List[str], vectors: List[List[float]]):
    with driver.session(database=NEO4J_DB) as session:
        for k, v in zip(keys, vectors):
            session.run(
                "MATCH (e:Encounter {key:$k}) SET e.%s = $v" % EMBEDDING_PROP,
                k=k, v=v
            ).consume()
# ___________________________________________________________________________________________________________________
# ПАЙПЛАЙН создания и записи всех ембедингов для Encounters
with get_driver() as driver:
    ensure_vector_index(driver)
    cards = fetch_encounter_cards(driver)
    keys, vecs = build_embeddings_for_encounters(cards)
    write_embeddings(driver, keys, vecs)
    print(f"Wrote embeddings for {len(keys)} encounters")


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 12203.08it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wrote embeddings for 154 encounters


# Поиск Encounters, наиболее близких к вопросу

In [25]:
# 2 Поиск top-k Encounter'ов наиболее близких семантически к данному вопросу

QUESTION = "Which documents do I need to bring to passport control in Paris?"
TOP_K = 50

# Build query embedding and retrieve nearest Encounters
with get_driver() as driver:
    # reuse existing model if present, otherwise load
    try:
        model
    except NameError:
        model = SentenceTransformer("intfloat/e5-base-v2")
    q_vec = embed_texts_e5(model, [QUESTION], is_query=True)[0]

    cypher = """
    CALL db.index.vector.queryNodes($index, $k, $vector)
    YIELD node, score
    WHERE 'Encounter' IN labels(node)
    RETURN coalesce(node.key, elementId(node)) AS encounter_key, score
    ORDER BY score DESC
    """
    rows = []
    with driver.session(database=NEO4J_DB) as session:
        rows = session.run(cypher, index=VECTOR_INDEX_NAME, k=TOP_K, vector=q_vec).data()

top_encounter_keys = [r['encounter_key'] for r in rows]
print(f"Top-{TOP_K} encounters: {len(top_encounter_keys)}")
print(top_encounter_keys[:10])


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 33179.62it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top-50 encounters: 50
['encounter_12270454_companion1_1', 'encounter_12100144_author_1', 'encounter_11709466_author_1', 'encounter_12204974_author_1', 'encounter_12265196_author_1', 'encounter_11870493_author_1', 'encounter_3812199_author_1', 'encounter_3902451_friend1_1', 'encounter_11383255_author_1', 'encounter_9343311_author_1']


# Добыча триплетов на глубине 2 от Encounter

In [28]:
# 3 fetch triples within depth<=2 from the retrieved encounters

def fetch_triples_depth2(driver, encounter_keys):
    if not encounter_keys:
        return []
    triples = []
    q1 = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[r]->(o)
    RETURN labels(e)[0] AS s_label, coalesce(e.key, elementId(e)) AS s_key,
           type(r) AS p, labels(o)[0] AS o_label, coalesce(o.key, elementId(o)) AS o_key
    """
    q2 = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[]->(n)-[r]->(o)
    RETURN labels(n)[0] AS s_label, coalesce(n.key, elementId(n)) AS s_key,
           type(r) AS p, labels(o)[0] AS o_label, coalesce(o.key, elementId(o)) AS o_key
    """
    q_doc = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[:hasDocumentCheck]->(:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
    WHERE di.documentType IS NOT NULL
    RETURN 'DocumentInstance' AS s_label, coalesce(di.key, elementId(di)) AS s_key,
           'documentType' AS p, 'Literal' AS o_label, toString(di.documentType) AS o_key
    """
    q_topic = """
    MATCH (e:Encounter) WHERE e.key IN $keys
    MATCH (e)-[:hasQuestioning]->(q:Questioning)
    WHERE q.topicKey IS NOT NULL
    RETURN 'Questioning' AS s_label, coalesce(q.key, elementId(q)) AS s_key,
           'topicKey' AS p, 'Literal' AS o_label, toString(q.topicKey) AS o_key
    """
    with driver.session(database=NEO4J_DB) as session:
        triples += session.run(q1, keys=encounter_keys).data()
        triples += session.run(q2, keys=encounter_keys).data()
        triples += session.run(q_doc, keys=encounter_keys).data()
        triples += session.run(q_topic, keys=encounter_keys).data()

    # de-duplicate
    seen = set()
    out = []
    for t in triples:
        key = (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])
        if key in seen:
            continue
        seen.add(key)
        out.append(t)
    return out

with get_driver() as driver:
    triples_depth2 = fetch_triples_depth2(driver, top_encounter_keys)

print('Depth<=2 triples:', len(triples_depth2))


Depth<=2 triples: 571


# Создание эмбедингов для триплетов

In [29]:
# 4a build embeddings for triples (simple approach)
# PLAN:
# 1) Convert each triple to a text string.
# 2) Embed all triple texts and the question.

def triple_to_text(t):
    return f"{t['s_label']} {t['s_key']} --{t['p']}--> {t['o_label']} {t['o_key']}"

triple_texts = [triple_to_text(t) for t in triples_depth2]

try:
    model
except NameError:
    model = SentenceTransformer('intfloat/e5-base-v2')

q_vec = embed_texts_e5(model, [QUESTION], is_query=True)[0]
t_vecs = embed_texts_e5(model, triple_texts, is_query=False)


# Фильтрация триплетов семантически 

In [30]:
# 4b semantic filtering of triples
TOP_N_TRIPLES = 200

import numpy as np
q = np.array(q_vec, dtype=np.float32)
T = np.array(t_vecs, dtype=np.float32)
q_norm = np.linalg.norm(q) + 1e-9
t_norm = np.linalg.norm(T, axis=1) + 1e-9
scores = (T @ q) / (t_norm * q_norm)

idx = np.argsort(-scores)[:min(TOP_N_TRIPLES, len(triples_depth2))]
triples_semantic = [triples_depth2[i] for i in idx]
print('Semantic triples:', len(triples_semantic))


Semantic triples: 200


# Подсчет метрик

In [ ]:
# 5

In [ ]:
# 5 metrics: precision/recall/F1 vs gold for question 1
import json
from pathlib import Path

def normalize_triple(t):
    # supports both flat triples and nested {s:{label,key}, p, o:{label,key}}
    if 's_label' in t:
        return (t['s_label'], t['s_key'], t['p'], t['o_label'], t['o_key'])
    # nested format
    s = t.get('s', {})
    o = t.get('o', {})
    return (s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key'))

def triples_to_set(triples):
    s = set()
    for t in triples:
        s.add(normalize_triple(t))
    return s

def load_q1_gold():
    base = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotation/q_1')
    triples = []
    for folder in ['fr_1','fr_2','germany','italy','spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            triples.extend(data.get('triples', []))
    return triples

retrieved_set = triples_to_set(triples_semantic)
gold_set = triples_to_set(load_q1_gold())
tp = len(retrieved_set & gold_set)
precision = tp / len(retrieved_set) if retrieved_set else 0.0
recall = tp / len(gold_set) if gold_set else 0.0
f1 = (2*precision*recall/(precision+recall)) if (precision+recall)>0 else 0.0
print('precision', precision)
print('recall', recall)
print('f1', f1)


# Создание эмбедингов для всех оставшихся узлов

In [ ]:
# N Embeddings for ALL non-Encounter nodes

from typing import Any

# Build text cards for every node except Encounter
# Uses elementId for stable lookup even if a node has no key property.

def fetch_non_encounter_cards(driver) -> dict:
    query = """
    MATCH (n)
    WHERE NOT n:Encounter
    RETURN elementId(n) AS eid, labels(n) AS labels, properties(n) AS props
    """
    cards = {}
    with driver.session(database=NEO4J_DB) as session:
        for r in session.run(query):
            eid = r["eid"]
            labels = r["labels"] or []
            props = r["props"] or {}
            parts = [f"Labels: {', '.join(labels)}"]
            if props.get('key') is not None:
                parts.append(f"Key: {props['key']}")
            # include a few common properties if present
            for k in [
                'entityName', 'documentType', 'topicKey', 'controlStage',
                'screeningStatus', 'outcome', 'checkMode',
                'presentedFormat', 'acceptedFormat'
            ]:
                if props.get(k) is not None:
                    parts.append(f"{k}: {props[k]}")
            cards[eid] = " | ".join(parts)
    return cards


def write_embeddings_by_element_id(driver, eids: list, vectors: list):
    with driver.session(database=NEO4J_DB) as session:
        for eid, v in zip(eids, vectors):
            session.run(
                f"MATCH (n) WHERE elementId(n) = $eid SET n.{EMBEDDING_PROP} = $v",
                eid=eid, v=v
            ).consume()


# Run embeddings for non-Encounter nodes
with get_driver() as driver:
    cards = fetch_non_encounter_cards(driver)
    if cards:
        keys = list(cards.keys())
        texts = [cards[k] for k in keys]
        # reuse existing model if present, otherwise load
        try:
            model
        except NameError:
            model = SentenceTransformer("intfloat/e5-base-v2")
        vecs = embed_texts_e5(model, texts, is_query=False)
        write_embeddings_by_element_id(driver, keys, vecs)
        print(f"Wrote embeddings for {len(keys)} non-Encounter nodes")
    else:
        print("No non-Encounter nodes found.")
